# **______________   STEAM PROJECT   _______________**

The goal of this project is to conduct a global analysis of the games available on Steam's marketplace in order to better understand the videogames ecosystem and today's trends. We want to understand what factors affect the popularity or sales of a video game to better prepare the release of a new revolutionary videogame. 

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StringType, LongType, DoubleType, ArrayType
import datetime

## **I. PREPARING DATA FOR ANALYSIS**

### **1. Importing dataset**

The dataset is stored in a S3 bucket. We will first extract it and see how it looks like with Databricks.

In [0]:
filepath = 's3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json'

In [0]:
raw_dataset = spark.read.format('json').load(filepath)

### **2. Main characteristics**

#### **2.1. Size of the dataset**

In [0]:
total_nb_rows = raw_dataset.count()
print (f" This dataset has {total_nb_rows} rows and {len(raw_dataset.columns)} columns.")

#### **2.2. Organization of the dataset**

In [0]:
raw_dataset.show(5) # Display the first 5 rows

We can see that this dataset is composed of 2 columns, one for the id (simple value) and the other one for the the data (nested format for each rows). Let's observe one row from the data column : 

In [0]:
raw_dataset.select(F.col("data")).take(1)

To better understand the structure of this nested data, we will analyze the schema of the dataset.

In [0]:
raw_dataset.printSchema() # Analyze how the dataset is structured

When we observe the schema of the dataset we notice that the variables are spread into several levels : 
- LEVEL N°1 : 
    - `data`=> STRUCT
    - `id`
- LEVEL N°2 : 
    - `appid`
    - `categories`=> ARRAY
    - `ccu`
    - `developer`
    - `discount`
    - `genre`
    - `header_image`
    - `initialprice`
    - `languages`
    - `name`
    - `negative`
    - `owners`
    - `platforms` => STRUCT
    - `positive`
    - `price`
    - `publisher`
    - `release_date`
    - `required_age`
    - `short_description`
    - `tags` => STRUCT
    - `type`
    - `website`
- LEVEL N°3 :
    - `element`
    - `linux`
    - `mac`
    - `windows`
    - all the tags
    

From this schema we can observe that : 
- we have one nested list (ARRAY) and 3 nested dictionaries(STRUCT). We will use the "dot notation" for the STRUCT and an "explode" for the array in order to "flatten" this nested file. 
- the information related to prices : `initialprice`, `discount`and `price` are strings. We will have to convert them into Float for further analyses.
- the information related to date `release_date` is a string and will have to be converted as a date format. 

#### **2.3. Get all nested columns**

We need to create a function to get all the nested columns. This function deals with simple columns, StructType and ArrayType. We need to escape special characters in the name of the columns to avoid parsing errors (for example : in the tags like `1980s`).

In [0]:
def get_nested_columns(schema, prefix=""):
    columns = []
    for field in schema.fields:
        # Escape field names
        valid_field_name = f"`{field.name}`"
        name = f"{prefix}.{valid_field_name}" if prefix else valid_field_name
        
        # Recursion if StructType
        if isinstance(field.dataType, StructType):
            columns += get_nested_columns(field.dataType, name)
        else:
            columns.append(name)
    return columns

In [0]:
all_column_names = get_nested_columns(raw_dataset.schema)
all_column_names

### **3. NULL values analysis**

In [0]:
# Count number of nulls for each column and calculate the percentage (we use * to transform the list into arguments for agg())
null_stats = raw_dataset.agg(*[
    F.round((F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) / F.lit(total_nb_rows) * 100.0), 2)
    .alias(c.replace(".", "_").replace("`", ""))
    for c in all_column_names
]).collect()[0]

# Transform into a DataFrame
null_df = spark.createDataFrame(
    list(null_stats.asDict().items()),
    ["column_name", "null_percentage"]
).orderBy(F.col("null_percentage").desc())

display(null_df)

From this table we can observe 2 main points : 
- There are no missing values for all the columns that are not tags
- There are high amounts of null values for tags. Here we can deduce that the null values represents games without the specific tag and not a missing information so we will replace them by 0 and instead of evaluating the percentage of nulls for the tags, we will calculate the frequency.

### **4. Drop useless columns**

#### **4.1. Drop rare tags**

In [0]:
# Replace NULL by 0 for tags
tag_columns = [c for c in all_column_names if "tags" in c]

df_clean = raw_dataset.fillna(0, subset=tag_columns)

In [0]:
# Analysis of the frequency of each tag (% of games with the tag)
tag_frequencies = {}
    
# % of games with the tag(> 0 votes)
freq_tag_row = df_clean.agg(*[
    F.round(
        F.sum(F.when(F.col(tag) > 0, 1)) / F.lit(total_nb_rows) * 100,
        2
    ).alias(tag.replace("`data`.`tags`.", ""))
    for tag in tag_columns
]).collect()[0]

# Transform into a DataFrame
freq_tag_df = spark.createDataFrame(
    list(freq_tag_row.asDict().items()),
    ["column_name", "frequency_percentage"]
).orderBy(F.col("frequency_percentage").desc())

display(freq_tag_df)

The top 5 tags (present in more than 20% of the games) are : 
- `Indie`
- `Singleplayer`
- `Action`
- `Casual`
- `Adventure`
 
 Moreover, a lot of tags are rare (present in less than 1% of all the games, we can drop them).

In [0]:
# Filter only rare tags (<1%)
threshold = 1
rare_tags_df = freq_tag_df.filter(F.col("frequency_percentage") < threshold)

# Create a list of rare tags
rare_tags = [row["column_name"] for row in rare_tags_df.collect()]

# Create corresponding names
rare_tag_columns = [f"`data`.`tags`.`{tag}`" for tag in rare_tags]

# Drop rare tags
df_clean = df_clean.drop(*rare_tag_columns)

# Stats
stats_df = freq_tag_df.agg(
    F.count("*").alias("nb_tot_tags"),
    F.sum(F.when(F.col("frequency_percentage") < 1, 1).otherwise(0)).alias("nb_rare_tags")
).withColumn(
    "percent_removed", F.round(F.col("nb_rare_tags") / F.col("nb_tot_tags") * 100, 2)
)

stats_df.display()

More than half of the tags are rare (<1%) and have been removed from the dataset.

#### **4.2. Drop non useful columns**

Some columns cannot be used for analysis (`id`, `appid`, `website`) because they are too diverse. Then `header_image` because we cannot study it.

In [0]:
# List of columns to drop
col_to_drop = ["id","data.appid","data.header_image", "data.website"]

# Drop rare tags
df_clean = df_clean.drop(*col_to_drop)

### **5. Prepare a cleaned dataset**

#### **5.1. Flatten the nested structure**

In [0]:
# We reuse the path names of the columns from the get_nested_columns function. We don't select the array "categories" so far, in order not too increase too much the number of rows. 
cols_to_select = [c for c in all_column_names if "categories" not in c]

# Flattenning 
df_flat = raw_dataset.select([
    F.col(c).alias(c.replace(".", "_").replace("`", "")) 
    for c in cols_to_select
])
df_flat.display()

#### **5.2. Formatting dates**

`release_date`is the only column concerning dates. It is a string format so we need to convert it into a date format. First, let's analyze the type of dates in the dataset.

In [0]:
df_flat.select("data_release_date").distinct().show(100, False)

We can observe several date formats : 
- yyyy/MM/dd
- yyyy/MM/d
- yyyy/MM 

We need to clean all the dates before converting the strings into a date format to avoid parsing errors. We will complete the dates without days by adding "01". We also use coalesce to deal with several kind of format.

In [0]:
df_flat_with_dates = (
    df_flat
    .withColumn(
        "cleaned_dates",
        F.when(
            F.col("data_release_date").rlike(r"^\d{4}/\d{2}$"),
            F.concat(F.col("data_release_date"), F.lit("/01"))
        )
        .when(
            F.col("data_release_date")=="",
            F.lit(None)
        )
        .otherwise(F.col("data_release_date"))
    )
    .withColumn(
        "parsed_date",
        F.coalesce(
            F.to_date(F.col("cleaned_dates"), "yyyy/MM/d"),
            F.to_date(F.col("cleaned_dates"), "yyyy/MM/dd"),
            F.to_date(F.col("cleaned_dates"), "yyyy/MM"),
            F.lit(None))
    ))
df_flat_with_dates.display(5)

Now we can extract years and months : 

In [0]:
df_flat_extracted_dates = df_flat_with_dates.withColumn(
    "release_year",
    F.year(F.col("parsed_date"))  # Extract year
).withColumn(
    "release_month",
    F.month(F.col("parsed_date"))  # Extract month
)
df_flat_extracted_dates.display()

#### **5.3. Cleaning types**

After dealing with date formats, we need to correct the wrong types we have in the dataset : 
- Information related to prices should be floats instead of strings (`price`, `initialprice`, `discount`).
- The `required_age`should be an integer instead of a string
To do so, we are using `cast()`.


In [0]:
# Convert the wrong types
df_flat_dates_types =  (
    df_flat_extracted_dates
    .withColumn(
        "data_price",
        F.col("data_price").cast("float")
    ) 
    .withColumn(
        "data_discount",
        F.col("data_discount").cast("float")
    )
    .withColumn(
        "data_initialprice",
        F.col("data_initialprice").cast("float")
    )
    .withColumn(
        "data_required_age",
        F.col("data_required_age").cast("int")
    )
)

In [0]:
# We check that there are no NULL created when the function cast() founds a wrong format.
df_flat_dates_types.select([
    F.count(F.when(F.col("data_price").isNull(), 1)).alias("price_null"),
    F.count(F.when(F.col("data_discount").isNull(), 1)).alias("discount_null"),
]).show()

#### **5.4. Adding relevant features**

Now that our dataset is flattened, that useless columns have been removed, and types corrected, we can now add relevant features for our analysis. The elements that will be necessary to judge the impact of some parameters are the clients'rates. 

We have `positive` and `negative` columns. With this columns, we can calculate several columns : `total_reviews`,`rating_score`. (% of positive reviews).

Then we can also create a boolean column `has_discount`.

In [0]:
# Total of reviews per game
df = df_flat_dates_types.withColumn(
    "total_reviews",
    F.col("data_positive") + F.col("data_negative")
)

# Calculate rating scores
df = df.withColumn(
    "rating_score",
    F.when(F.col("total_reviews") > 0,
         F.col("data_positive") / F.col("total_reviews")
    ).otherwise(None)  # Null if no reviews
)

# Create boolean column
df = df.withColumn(
    "has_discount",
    F.col("data_discount") > 0  # 1 = discount, 0 = no discount
)

In [0]:
# Print schema for verification
df.printSchema()

## **II. DATA ANALYSIS**

To answer busisness questions to understand the main trends of the video game industry, we will divide the analysis into () main axis : 
- Publications (top and flop publishers, evolution of publications over time, etc.)
- Games (top and flop games, languages, restricted ages, etc.)
- Genres (favorite genres, most lucrative genres)
- Prices (stats, distribution and evolution of evolution of prices)
- Technologies

### **1. Publications analysis**

#### **1.1. Publishers main information**

We can first analyse the main characteristics of publisher to answer the following questions : 
- How many publishers does Steam have? 
- How many publications does every publisher have?
- Which are the best or the worst publishers?
- How the number of publications evolved over time?

In [0]:
# Total number of publishers
total_nb_publishers = df.select(F.col("data_publisher")).distinct().count()
print(f" Total number of publishers : {total_nb_publishers}")

In [0]:
# Number of publications per publisher
nb_publications_per_publisher = (
    df
    .groupBy(F.col("data_publisher"))
    .count()
).orderBy(F.desc("count"))

display(nb_publications_per_publisher)

From this table we can see that : 
- The highest number of publications per publisher is 422 and it has been reached by "Big Fish Games"

- If we consider the number of publications, the top 5 publishers are : 
    - Big Fish Games
    - 8floor
    - SEGA
    - Strategy First
    - Square Enix
- A high number of publishers are low publishers with only one publication. 

The graphic shows us that some publishers have a huge impact on the market while the vast majority is represented by small publishers.

Let's see how many publishers have only one publication.

In [0]:
# Number of publishers with only one publication
last_publishers = (
    nb_publications_per_publisher
    .select("data_publisher")
    .where(F.col('count')==1)
)
last_publishers_nb = last_publishers.count()

print(f"Out of {total_nb_publishers} publishers, {last_publishers_nb} have only one publication. This represents {round(last_publishers_nb/total_nb_publishers*100,2)}% of the total.")

# Visualization
pie_data = spark.createDataFrame(
    [
        ("Single publication", last_publishers_nb),
        ("Multiple publications", total_nb_publishers-last_publishers_nb)
    ],
    ["category", "nb_publishers"]
)

pie_data.display()

In [0]:
# Mean number of releases per publisher
nb_publications = df.count()
avg_publications = round(nb_publications/total_nb_publishers,2)

print(f"On average, publishers have done {avg_publications} publications on Steam.")

#### **1.2. Publishers' reviews**

We have compared publishers in terms of number of publications, let's see now the ones that are best rated. Are there games that were really well rated with only a few publications? Preferring quality over quantity? For that, we will compare ratios of positive rates instead of total numbers of rates. 

In [0]:
reviews_per_publisher = (
    df
    .groupBy("data_publisher")
    .agg(
        F.sum("data_positive").alias("total_positive_reviews"),
        F.sum("data_negative").alias("total_negative_reviews"),
        F.sum("total_reviews").alias("total_reviews_per_publisher"),
        F.round(100*F.avg("rating_score"),2).alias("avg_rating_score (%)"),
        F.count("data_publisher").alias("nb_publications"))
    .orderBy(F.desc(F.col("avg_rating_score (%)"))
    )
)
reviews_per_publisher.display()

We ordered the publishers according to their average rate score. We can observe that a lot of small publishers with only one publication have a high rate score. This is because they have only a few positive reviews and no negative reviews.

We need to conduct a deeper analysis to take into considaration not only the average rate score but also the number of publications so that it better reflects reality and business needs.

To do so, we can use the Bayesian score to weight the positive ratio by the total number of votes, to prevent publishers with very few votes from artificially appearing at the top.

**Formula**: 
$$
bayesian-score = \frac{v}{v+m}R + \frac{m}{v+m}C
$$

Where:

- R = average rating of the publisher  
- v = total number of reviews per publisher 
- m = minimum number of reviews required  
- C = global average rating across all publishers


This formula shrinks scores with few reviews toward the global mean, producing a more reliable ranking.

Publishers with zero reviews were excluded from the ranking, as their ratings contain no information about user perception.  
Since the Bayesian score relies on the number of reviews to estimate reliability, publishers without reviews would simply receive the global average score and therefore provide no meaningful ranking signal.

In [0]:
# Calculate the number of publishers with no reviews
no_review_count = reviews_per_publisher.filter(
    F.col("total_reviews_per_publisher") == 0
).count()

print(f"There are {no_review_count} publishers with no reviews at all (it represents {round(no_review_count/total_nb_publishers*100,2)}% of all publishers).")


We now remove them : 

In [0]:
# Drop publishers with no reviews
reviews_per_publisher = reviews_per_publisher.filter(
    F.col("total_reviews_per_publisher") > 0
)

In [0]:
# Calculate "C" : global average ration across all publishers
global_avg = (
  reviews_per_publisher
  .groupBy()
  .agg(
      F.avg("avg_rating_score (%)").alias("global_mean")
  )
).collect()[0]['global_mean']

print(f"The average rating score is {round(global_avg,2)}%")

Now let's identify "m" as the 80% percentiles of reviews : 

In [0]:
# Calculate "m" (we will fix "m" as the number of reviews at the 80th percentile)
m = (
    reviews_per_publisher
    .select(F.percentile_approx(F.col("total_reviews_per_publisher"), 0.80).alias("m"))
    .collect()[0]['m']
)

print(f"m = {m}")


In [0]:
# Add Bayesian rating score into the DataFrame
reviews_per_publisher = (
    reviews_per_publisher
    .withColumn(
        "bayesian_score",
        (
            (F.col("total_reviews_per_publisher") /
            (F.col("total_reviews_per_publisher") + F.lit(m))) * F.col("avg_rating_score (%)")
            +
            (F.lit(m) /
            (F.col("total_reviews_per_publisher") + F.lit(m))) * F.lit(global_avg)
        )
    )
)

In [0]:
# Top 5 publishers with the highest Bayesian rating score
top_5_bayesian = (
    reviews_per_publisher
    .orderBy(F.desc(F.col("bayesian_score")))
    .limit(5)
    .display()
)


The top 5 publishers according to their reviews are :
- adamgryu
- Igara Studio
- Studio Minus
- poncle
- HIKARI FIELD, NekoNyan Ltd.

They are small publishers with one or two games released, but those games have been positively reviewed by more than 10 000 players. It means that some of the small publishers can have a real positive impact for Steam.

In [0]:
# Bottom 5 publishers with the lowest Bayesian rating score
last_5_bayesian = (
    reviews_per_publisher
    .orderBy(F.asc(F.col("bayesian_score")))
    .limit(5)
    .display()
)

The bottom 5 publishers according to their reviews are :
- Asylum Entertainment Inc.
- Badland Studio
- Dark Day Interactive
- Atari, RCTO Productions
- 22cans Ltd.

They are small publishers as well with one or two games released, but those games have a higher proportion of negative reviews. 

Some small publishers can be really appreciated by the players, others are not, we will understand later what seduces players in a game. 

In [0]:
# Bayesian score of the publishers with the highest number of publications 
(
    reviews_per_publisher
    .orderBy(F.desc(F.col("nb_publications")))
    .limit(5)
    .display()
)

The five main publishers with have volumes of publications that we identified earlier have a Bayesian score above 70% except for "Strategy First" with a score of 56%.

#### **1.3. Time vs publications**

**_1.3.1. Evolution of the number of publications per year_**

Now, let's study the evolution of the publications over time. 
Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

In [0]:
releases_per_year = (
    df
    .groupBy("release_year")
    .count()
    .orderBy("release_year")
)

releases_per_year.display()

We can observe the following trends in the number of game releases:

- **1997–2013:** Slow but steady growth in releases, with the number of releases gradually increasing from 2 to around 500.  
- **2013–2018:** Rapid exponential growth, reaching a peak of more than 7,600 releases in 2018.  
- **2019:** Slight decrease compared to 2018, signaling a slowdown after the exponential growth phase (dropped of about 700 releases).
- **2020-2021:** Recovery with a new peak, likely due to market adjustments after the previous slowdown.  
- **2022:** New decline observed (dropped to 7400 releases), potentially influenced by external factors such as COVID-19 disruptions in 2020, which may have delayed development and publishing schedules.

In [0]:
top_year = (
    releases_per_year
    .select(
        F.col("release_year"),
        F.col("count").alias('nb_releases')
        )
    .orderBy(F.desc(F.col("nb_releases")))
    .limit(1)
)
top_year.display()

The year with the greater amount of release is 2021 with 8823 releases. 

**_1.3.2. Distribution of releases per month_**

Are there month or period of the years with a greater amount of publications? (e.g. before Christmas?).

In [0]:
releases_per_month = (
    df
    .groupBy("release_month")
    .count()
    .orderBy("release_month")
)

releases_per_month.display()

The number of releases over the months is not constant, we can see : 
- 3 pics (March and May with around 4700 releases and a major pic in October (more than 5500 releases))
- the weakest period of the year in term of releases are January, December and June. 

We can suggest that releases are mostly done in October for publishers to boost their sales before Christmas, and that there are less publications in January, December and June because of holidays and a slow down of the business activity. 

#### 1.4. Insights about publications analysis

### **2. Games analysis**

#### 2.1 Games' reviews

In [0]:
nb_games = df.count()
print(f"There are {nb_games} games in the dataset.")

As for the publishers, we can calculate the Bayesian score for each game allowing us to ponderate the rate ratio by the number of reviews. 

The formula is the same but this time we have : 
- R : games individual ratings
- v : number of reviews for an individual game
- m : minimum number of reviews required to trust a game'm rating fully
- C : global average rating across all games in the dataset

First we need to remove games with no publications at all.

In [0]:
# Calculate the number of games with no reviews
no_review_count = df.filter(
    F.col("total_reviews") == 0
).count()

print(f"There are {no_review_count} game(s) with no reviews at all (it represents {round(no_review_count/nb_games*100,2)}% of all publishers).")


We now remove them : 

In [0]:
# Drop publishers with no reviews
df = df.filter(
    F.col("total_reviews") > 0
)

In [0]:
# Calculate "C":
global_avg_games = df.agg(F.avg("rating_score").alias("global_avg")).collect()[0]["global_avg"]

print(f"The average rating score is {global_avg_games:.2f}")

In [0]:
# Calculate "m" (we will fix "m" as the number of reviews at the 80th percentile)
m = (
    df
    .select(F.percentile_approx(F.col("total_reviews"), 0.80).alias("m"))
    .collect()[0]['m']
)

print(f"m = {m}")


In [0]:
df = df.withColumn(
    "bayesian_score",
    (F.col("total_reviews") / (F.col("total_reviews") + F.lit(m))) * F.col("rating_score") +
    (F.lit(m) / (F.col("total_reviews") + F.lit(m))) * F.lit(global_avg_games)
)

In [0]:
top_5_games = df.orderBy(F.desc("bayesian_score")).select(
    "data_name","data_publisher","bayesian_score", "total_reviews"
).limit(5)
top_5_games.display()

The best games are : 
- People Playground	_(Studio Minus)_
- Aseprite	_(Igara Studio)_
- Portal 2	_(Valve)_
- A Short Hike _(adamgryu)_
- Vampire Survivors	_(poncle)_

We found almost the same results than for the best publishers analysis before, that is understandable because they had only one publication, so they published successful games that we see here. The only difference is "Portal 2" from Valve, that was not in our list of top five publishers. 

In [0]:
reviews_per_publisher.filter(F.col("data_publisher") == "Valve").display()

The publisher Valve has 33 publications and the greatest amount of positive reviews (but a great amount of negatives reviews as well) explaining its bayesian score.

#### 2.2. Insights about games analysis

### **3. Genre analysis**

### **4. Prices analysis**

We can wonder what is the effect of prices on the number of reviews : What is the general distribution of prices? Are top games more expensive or cheaper? Are games with a high proportion of negative reviews cheaper? Are big publishers offering more discounts? etc. There are 3 variables giving us information about prices : 
- `discount`
- `initialprice`
- `price`
If there is no discount : `initialprice` = `price`.

#### 4.1. Prices main information

**_4.1.1. General prices stats_**

In [0]:
# Selecting interesting columns
price_info = df.select(
    F.col("data_name"),
    F.col("data_publisher"),
    F.col("data_initialprice"),
    F.col("data_discount"),
    F.col("data_price"),
    F.col("data_positive"),
    F.col("data_negative"),
    F.col("total_reviews"),
    F.col("rating_score"),
    F.col("has_discount"),
    F.col("bayesian_score"),
    F.col("release_year"),
    F.col("release_month")
)
price_info.display()

In [0]:
# Distribution of all prices
prices_distribution = (
    price_info
    .groupBy("data_price")
    .count()
    .orderBy(F.desc("count"))
)
prices_distribution.display()

We can see than more than 700 games have a price = 0, Steam offers free games, the economic model of these game is different from the others so we will split the dataset into 2 categories. 

In [0]:
free_games = price_info.filter(F.col("data_price") == 0)
paid_games = price_info.filter(F.col("data_price") > 0)

In [0]:
# We verify that all free games are initially free and not discounted to free
tot_free = len(price_info.filter(F.col("data_price") == 0).collect())
tot_initial_free = len(price_info.filter(F.col("data_initialprice") == 0).collect())

tot_free == tot_initial_free

In [0]:
# Distribution of paid_games only
paid_games_distribution = (
    paid_games
    .groupBy("data_price")
    .count()
    .orderBy(F.desc("count"))
)
paid_games_distribution.display()

The prices follow a long-tail distribution, with most games priced between 28 and 7000 of the currency used for the dataset.

In [0]:
stat_prices_paid_games =(
    paid_games
    .agg(
        F.round(F.avg("data_initialprice"),2).alias("avg_initialprice"),
        F.round(F.avg("data_discount"),2).alias("avg_discount"),
        F.min("data_discount").alias("min_discount"),
        F.max("data_discount").alias("max_discount"),
        F.round(F.avg("data_price"),2).alias("avg_price"),
        F.min("data_price").alias("min_price"),
        F.max("data_price").alias("max_price"),
        F.count("*").alias("nb_games"),
    )
)
stat_prices_paid_games.display()

These statistics are giving us the following information : 
- median price : 599 
- High percentil (99%) : 4999

In [0]:
# Identification of expensive games (>99th percentile)
percentil_99 = paid_games.approxQuantile("data_price", [0.99], 0)[0]
print(f"99th percentile price: {percentil_99}")

# Filter most expensive games
most_expensive_games = (
    price_info
    .filter(F.col("data_price") > percentil_99)
)
most_expensive_games.display()

print(f"There are {most_expensive_games.count()} games with price > {percentil_99} (99th percentile)")

These expensive games must represent collector editions and special offers

Now that we have observed the distribution of prices we are going to segment the paid_games dataframe into several categories using price percentiles to ensure balanced groups across segments despite the highly right-skewed distribution. Here are the segments : 
- 0–25 %	very cheap games
- 25–50 %	cheap games
- 50–75 %	mid-range games
- 75–90 %	expensive games
- 90–100 %	premium games

In [0]:
# Calculate quantiles
quantiles = paid_games.approxQuantile(
    "data_price",
    [0.25, 0.5, 0.75, 0.9],
    0
)

p25, p50, p75, p90 = quantiles

In [0]:
# Segmentation of the prices
paid_games = paid_games.withColumn(
    "price_segment",
    F.when(F.col("data_price") <= p25, "very_low_price")
    .when(F.col("data_price") <= p50, "low_price")
    .when(F.col("data_price") <= p75, "mid_price")
    .when(F.col("data_price") <= p90, "high_price")
    .otherwise("premium_price")
)

In [0]:
# Distribution of games according to the segmentation
paid_games_segmentation = (
    paid_games
    .groupBy("price_segment")
    .count()
    .orderBy(F.desc("count"))
)
paid_games_segmentation.display()

We can observe that more than a quarter of all games have a very low price, almost half of them have a low to mid price and the last quarter have a price from high to premium.

**_4.1.2. Discounts analysis_**

What is the proportion of discounted games?

In [0]:
nb_total = price_info.count()

# Ratio of discounted games
discount_ratio = (
    price_info
    .agg(
        (
            F.sum((F.col("data_discount") != 0).cast("int")) /
            F.count("*")
        ).alias("discount_ratio")
    )
)

ratio = discount_ratio.collect()[0]["discount_ratio"]

print(f"Only {100 * ratio:.2f}% of games are discounted.")

What are the main discounted applied?

In [0]:
# Distribution of discounts
discount_distribution = (
    price_info
    .filter(F.col("data_discount") != 0)
    .groupBy("data_discount")
    .count()
    .orderBy(F.desc("count"))
)
discount_distribution.display()

The main discount percentages applied are tens (50% then 90%,80%) etc. 

Which games are mostly concerned by discounts? Is it the most expensive ones?

In [0]:
# Proportion of discounted games per segment
discount_per_segment = (
    paid_games
    .groupBy("price_segment")
    .agg(
        (
            F.sum((F.col("data_discount") != 0).cast("int")) /
            F.count("*")
        ).alias("%_discounted_games")
    )
)
display(discount_per_segment)

Almost half of the very low prices games are concerned by discounts. Whereas less than 7% of premium prices games are discounted.

#### 4.2. Prices according to publishers

Are prices dependent of the size of the publisher ? Are some publishers offering more discounts than others? Are some publishers offering more free games than others?

In [0]:
stat_price_per_publisher = (
    price_info
    .groupBy("data_publisher")
    .agg( 
        F.round(F.avg("data_initialprice"),2).alias("avg_initialprice"),
        F.round(F.avg("data_discount"),2).alias("avg_discount"),
        F.min("data_discount").alias("min_discount"),
        F.max("data_discount").alias("max_discount"),
        F.round(F.avg("data_price"),2).alias("avg_price"),
        F.min("data_price").alias("min_price"),
        F.max("data_price").alias("max_price"),
        F.count("*").alias("nb_games")
    ))

stat_price_per_publisher.display(5)

#### 4.3. Prices evolution over time

Have the prices evolved over time? 
Are there periods of the year with more discount? 
Are there periods of the year where the games are best sold? 

**_4.3.1. Evolution of prices over years_**

In order to see if the prices evolved over time we will see the evolution of several percentiles : 5% / 50% / 95%. This will allow us to see the principal trends (increase, decrease, rise of cheap or really expensive games, etc.)

In [0]:
price_trend_years = (
    price_info
    .groupBy("release_year")
    .agg(
        F.expr("percentile(data_price, 0.5)").alias("median_price"),
        F.expr("percentile(data_price, 0.05)").alias("p05_price"),
        F.expr("percentile(data_price, 0.95)").alias("p95_price")
    )
    .orderBy("release_year")
)

price_trend_years.display()

We can observe the following trends in prices over time : 

- 1997 to 2005 : Prices are evolving more often during this period of time, than can be explained by the small amount of sales where a different price on a game had a biger impact. The average prices dropped between 1997 to 1999 (from about 1500 to 500) and increased back to the initial price in 2002.
- From 2006 : 
    - p05% : There are almost no more really cheap games (p05) until 2021 where it started to increase a bit again. This can be explained by the fact that the company was growing rapidely and didn't need to offer cheap prices anymore compared to its early stages. 
    - p95% : We also have a stabilization of really high priced games during that period, with pics in 2012, 2013 and 2019. Then after a drop in 2020 a new increased is observed from 2020 to 2022.


**_4.3.2. Evolution of prices over month_**

Are there period of the year with different prices (more discounts, etc.)?

In [0]:
# Distribution of prices over months
price_trend_months = (
    price_info
    .groupBy("release_month")
    .agg(
        F.percentile_approx("data_price", 0.5).alias("median_price"),
        F.percentile_approx("data_price", 0.05).alias("p05_price"),
        F.percentile_approx("data_price", 0.95).alias("p95_price")
    )
    .orderBy("release_month")
)

display(price_trend_months)

#### 4.4. Insights about prices analysis

### **5. Technologies analysis**

## **III. CONCLUSIONS**